# Suite EXE — `txn.exec`: a transaction's suffix over a stream of frames, read-only or writing, through the loopback

Within a transaction, `with store.transaction() as txn:`, the transaction open, `txn`, executes the suffix of a
transaction's expression over input frames with `txn.exec(frame, remainder, allow_write)`, giving a stream of frames,
a generator in Python, and opening and committing no transaction of its own
([CAP-308: stacking by mutation transactions, in one round trip](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-308); [the stacked stores proposal, a cache layer and the layer below](https://github.com/pitaman71/mbse-basis/blob/main/proposals/STACKED-STORES.md#a-cache-layer-and-the-layer-below),
its first stage; the first piece of [the transaction-object proposal](https://github.com/pitaman71/mbse-schemas/blob/main/docs/proposals/transaction-object/README.md)). Writing, it interprets the expression
fully, as `store.transaction` does, which collects its stream over one empty input frame. Read-only, it interprets
over what the store holds: where it holds nothing to bind a symbol, or a term would write, the frame halts and is given
carrying its `remainder`, a position in the original expression, which the same transaction completes, writing: the
loopback, a store with no layer below. A partition keeps its join, the rest of a branch a frame halted in executed by
the loopback. Each frame carries the id of the input frame it descends from, `scope`, and each input frame's frames
end with its completion marker, `{"end": scope}`; errors are annotated on frames ([CAP-309: errors annotated on frames](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/README.md#CAP-309)), and frames hold
references, not visitors ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md#decided)). Everything commits, or is dropped, with the scope.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
from mbse.Schemas.Framework import Builders as B, Changes, Plain, References, Transactions
from mbse.Schemas.Framework.Transactions import Frame, Remainder
from support import raises


def crm(txn):
    """crm.Person, owned by the singleton crm.TheBook under a key, who knows persons; crm.Note, counted."""
    S = txn.Schemas
    return txn.uid("crm/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Knows": S.OfRelation, "Person": S.OfObject, "Holds": S.OfRelation, "Book": S.OfObject,
                  "Note": S.OfObject})
        .spec([
            lambda _: _.Knows.name("crm.Knows").links([lambda t: t.name("knower"), lambda t: t.name("known")]),
            lambda _: _.Person.name("crm.Person").ref(True).properties({"name": S.OfStr, "age": S.OfInt}).adjacencies([
                lambda t: t.name("knows").relation([_.Knows]).me("knower"),
                lambda t: t.relation([_.Knows]).me("known"),
                lambda t: t.name("book").relation([_.Holds]).me("contact").owner(True),
            ]),
            lambda _: _.Holds.name("crm.Holds").links([lambda t: t.name("book"), lambda t: t.name("contact")]).properties({
                "key": S.OfStr}).uniques([["book", "key"], ["contact"]]),
            lambda _: _.Book.name("crm.Book").ref(True).singleton("crm.TheBook").adjacencies({
                "contacts": lambda t: t.relation([_.Holds]).me("book")}),
            lambda _: _.Note.name("crm.Note").ref(True).properties({"text": S.OfStr}),
        ])]))])


def made():
    """A store holding Ada, 36, who knows Bob, and Bob, of no age: crm.TheBook's contacts "ada" and "bob"."""
    store = B.Store()
    store.transaction(crm)
    Person, Book = store.registered("crm.Person"), store.registered("crm.Book")
    store.transaction(lambda t: t.uid("people/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
        {"a": Person, "b": Person, "book": Book}).spec([
            lambda _: _.book.contacts({"ada": lambda e: e.contact(_.a), "bob": lambda e: e.contact(_.b)}),
            lambda _: _.a.name("Ada").age(36).knows([_.b]),
            lambda _: _.b.name("Bob"),
        ])]))]))
    return store


def frames(items):
    """The frames of a stream's items, its completion markers left out."""
    return [item for item in items if isinstance(item, Frame)]


def designated(store, reference):
    """The person a reference names, a visitor of the transaction open, by its key in crm.TheBook's contacts; None for
    the book itself."""
    path = Plain.ToPlain(store)(reference.as_schema(), reference)["path"]["next"]
    return store.singleton("crm.TheBook").contacts(path["entry"]["properties"]) if "entry" in path else None


def read_names(store, given):
    """The names of the persons each frame binds, by symbol, the book as "book", read within the transaction open by
    their references."""
    return [{symbol: "book" if found is None else str(found.name)
             for symbol, found in ((symbol, designated(store, target)) for symbol, target in frame.items())}
            for frame in given]


def names(store, given):
    """`read_names`, within a transaction of its own."""
    with store.transaction():
        return read_names(store, given)


def read_people(store):
    """Each person's name, age and whom they know, read within the transaction open."""
    return sorted((str(p.name), store.member(p, "age"), sorted(str(e.known.name) for e in p.knows))
                  for p in store.extent("crm.Person"))


def people(store):
    """`read_people`, within a transaction of its own."""
    with store.transaction():
        return read_people(store)


def run(store, frame, remainder, allow_write):
    """The items of `txn.exec`'s stream, within a transaction of their own, which commits."""
    with store.transaction() as txn:
        return list(txn.exec(frame, remainder, allow_write))


def ada_then(then):
    """A transaction: a map binding Ada to x, then the term `then` gives."""
    def build(txn):
        Person = txn.crm.Person
        return txn.uid("ada/1").mutations([lambda mu: mu.map(lambda m: m.symbols([
            lambda p: p.symbols({"x": Person}).spec([lambda _: _.x.name("Ada")])]).then([then(Person)]))])
    return build


def aged(age):
    """An update writing x's age."""
    return lambda Person: lambda w: w.update(lambda u: u.apply([
        lambda p: p.symbols({"x": Person}).spec([lambda _: _.x.age(age)])]))


def met(name):
    """A map binding to y the person named `name`."""
    return lambda Person: lambda w: w.map(lambda m: m.symbols([
        lambda p: p.symbols({"x": Person, "y": Person}).spec([lambda _: _.y.name(name)])]))

<a id="EXE-01"></a>
## EXE-01 · The transaction open, `txn`, its `exec` writing over one empty input frame, is `store.transaction`: the same frames and writes, committed with the scope, each frame carrying its input frame's id, then that input frame's completion marker; over a stream of input frames, each binding symbols by their references, each input frame's frames, then its marker

In [ ]:
one, two = made(), made()
with one.transaction() as txn:
    assert isinstance(txn, Transactions.Transaction)
    items = list(txn.exec(Frame(), ada_then(aged(37)), True))
    assert read_people(one) == [("Ada", 37, ["Bob"]), ("Bob", None, [])]  # written within it
assert len(items) == 2 and items[1] == {"end": "0"}  # its frame, then its completion marker
(done,) = frames(items)
assert done.scope == "0" and done.remainder is None and done.errors == [] and done.assigns == {"x": {"name": "Ada", "age": 37}}
given = two.transaction(ada_then(aged(37)))  # its transaction's `exec`, collected
assert names(one, [done]) == names(two, given) == [{"x": "Ada"}] and given[0].scope == "0"
assert people(one) == people(two) == [("Ada", 37, ["Bob"]), ("Bob", None, [])]
with one.transaction():  # it gave references, not visitors (CAP-315)
    assert designated(one, done["x"]) is one.singleton("crm.TheBook").contacts({"key": "ada"})
everyone = one.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([
    lambda p: p.symbols({"x": t.crm.Person})]))]))
older = lambda t: t.uid("older/1").mutations([lambda mu: mu.update(lambda u: u.apply([
    lambda p: p.symbols({"x": t.crm.Person}).spec([lambda _: _.x.age(50)])]))])  # x bound by each input frame
items = run(one, [{"x": frame["x"]} for frame in everyone], older, True)  # plain mappings: their positions
assert [item.scope if isinstance(item, Frame) else item for item in items] == ["0", {"end": "0"}, "1", {"end": "1"}]
assert names(one, frames(items)) == [{"x": "Ada"}, {"x": "Bob"}] and people(one) == [
    ("Ada", 50, ["Bob"]), ("Bob", 50, [])]
items = run(one, Frame({"x": everyone[1]["x"]}, scope="bob"), older, True)  # a frame's own scope, its id
assert items[0].scope == "bob" and items[1] == {"end": "bob"}
with raises(ValueError, match="symbol 'x' of an update is bound by no term before it"):
    one.transaction(older)  # no input frame binds x
with raises(ValueError, match="stop"):
    with one.transaction() as txn:
        list(txn.exec(Frame(), ada_then(aged(99)), True))
        raise ValueError("stop")  # the scope raises: what exec wrote is dropped with it
assert people(one) == [("Ada", 50, ["Bob"]), ("Bob", 50, [])]
with one.transaction() as txn:
    kept = txn.exec([Frame()], ada_then(aged(98)), True)
with raises(ValueError, match="a transaction's exec runs within it: this one has ended"):
    next(kept)  # a stream used after its transaction

<a id="EXE-02"></a>
## EXE-02 · Read-only: interpreted over what the store holds, writing nothing; a frame halts where a term would write, or where the store holds nothing to bind a symbol, given as bound so far, carrying its remainder, a position in the original expression; the same transaction, given it, completes it writing (the loopback), as a full run does, all committed with the scope

In [ ]:
one, two = made(), made()
older = ada_then(aged(40))
with one.transaction() as txn:
    items = list(txn.exec(Frame(), older, False))
    (halted,) = frames(items)  # Ada is 36: the update would write
    assert read_people(one) == [("Ada", 36, ["Bob"]), ("Bob", None, [])]  # nothing written
    looped = frames(txn.exec(halted, halted.remainder, True))  # the loopback: the same transaction, writing
    assert read_people(one) == [("Ada", 40, ["Bob"]), ("Bob", None, [])]
assert items[1] == {"end": "0"} and names(one, [halted]) == [{"x": "Ada"}] and halted.scope == "0"
remainder = halted.remainder
assert remainder.at == ("mutations", 0, "map", "then", 0, "update") and remainder.subterm == "$.mutations[0].map.then[0].update"
assert repr(remainder) == "<remainder $.mutations[0].map.then[0].update>"
assert remainder == Remainder(remainder.transaction, list(remainder.at)) and hash(remainder) == hash(remainder.at)
assert remainder != Remainder(remainder.transaction, ("mutations", 0, "map")) and remainder != "elsewhere"
full = two.transaction(older)
assert names(one, looped) == names(two, full) == [{"x": "Ada"}] and people(one) == people(two) == [
    ("Ada", 40, ["Bob"]), ("Bob", None, [])]
assert frames(run(one, Frame(), older, False))[0].remainder is None  # what holds is read, and given whole
cy = lambda t: t.uid("cy/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"c": t.crm.Person, "book": t.crm.Book}).spec([lambda _: _.book.contacts({"cy": lambda e: e.contact(_.c)}),
                                                  lambda _: _.c.name("Cy")])]))])
with one.transaction() as txn:
    (halted,) = frames(txn.exec(Frame(), cy, False))  # an upsert that would create: it halts as it was given to it
    assert dict(halted) == {} and halted.remainder.at == ("mutations", 0, "upsert") and len(read_people(one)) == 2
    looped = frames(txn.exec(halted, halted.remainder, True))
full = two.transaction(cy)
assert names(one, looped) == names(two, full) and names(one, looped)[0]["c"] == "Cy" and people(one) == people(two)
with one.transaction() as txn:
    items = list(txn.exec(Frame(), ada_then(met("Dan")), False))  # nothing the store holds binds y
    (halted,) = frames(items)
    assert read_names(one, [halted]) == [{"x": "Ada"}] and halted.remainder.subterm == "$.mutations[0].map.then[0].map"
    assert frames(txn.exec(halted, halted.remainder, True)) == []  # none, as a full run
assert two.transaction(ada_then(met("Dan"))) == []
note = lambda t: t.uid("note/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"n": t.crm.Note}).spec([lambda _: _.n.text("x")])]))])
retext = lambda t: t.uid("retext/1").mutations([lambda mu: mu.update(lambda u: u.apply([lambda p: p.symbols(
    {"n": t.crm.Note}).spec([lambda _: _.n.text("y")])]))])
with one.transaction() as txn:  # a counted object a frame binds, which has no reference, resumed within the transaction
    (noted,) = frames(txn.exec(Frame(), note, True))
    (halted,) = frames(txn.exec(noted, retext, False))  # its text would be written
    assert halted.remainder.at == ("mutations", 0, "update")
    (resumed,) = frames(txn.exec(halted, halted.remainder, True))
    assert resumed.errors == [] and resumed.remainder is None
with raises(LookupError, match="symbol 'n' of the frame: a counted 'crm.Note' has no reference"):
    resumed["n"]  # after it, refused, as every frame's counted object is

<a id="EXE-03"></a>
## EXE-03 · A halt within a term: a map that binds some of its symbols, and nothing the store holds binds the rest, halts each frame as bound so far, its remainder that map and the terms after it; given again, the map binds the rest alone; a symbol given a reference that designates nothing halts alike

In [ ]:
one = made()
pair = lambda t: t.uid("pair/1").mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols(
    {"x": t.crm.Person, "y": t.crm.Person}).spec([lambda _: _.y.name("Cy")])]).then([
        lambda w: w.update(lambda u: u.apply([lambda p: p.symbols({"x": t.crm.Person, "y": t.crm.Person}).spec([
            lambda _: _.x.knows([_.y])])]))]))])
cy = lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"c": t.crm.Person, "book": t.crm.Book}).spec([lambda _: _.book.contacts({"cy": lambda e: e.contact(_.c)}),
                                                  lambda _: _.c.name("Cy")])]))])
with one.transaction() as txn:
    items = list(txn.exec(Frame(), pair, False))
    halted = frames(items)  # x bound to each person, y to none: each halts within the map
    assert read_names(one, halted) == [{"x": "Ada"}, {"x": "Bob"}] and items[-1] == {"end": "0"}
    assert {frame.remainder.subterm for frame in halted} == {"$.mutations[0].map"}
    list(txn.exec(Frame(), cy, True))  # Cy, written meanwhile
    looped = [frame for each in halted for frame in frames(txn.exec(each, each.remainder, True))]
    assert read_names(one, looped) == [{"x": "Ada", "y": "Cy"}, {"x": "Bob", "y": "Cy"}]  # x as it was bound; y, the rest
assert people(one) == [("Ada", 36, ["Bob", "Cy"]), ("Bob", None, ["Cy"]), ("Cy", None, [])]
bob = one.transaction(ada_then(met("Bob")))[0]["y"]  # Bob's reference
with one.transaction():
    one.crm.Book(one.singleton("crm.TheBook"), lambda b: b.contacts(lambda a: a.remove(lambda e: e.key("bob"))))
named = lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"b": bob})]))])
with one.transaction() as txn:
    (halted,) = frames(txn.exec(Frame(), named, False))  # Bob deleted: his reference designates nothing
    assert dict(halted) == {} and halted.remainder.at == ("mutations", 0, "map")
    assert frames(txn.exec(halted, halted.remainder, True)) == []  # none, as a full run

<a id="EXE-04"></a>
## EXE-04 · Split-join through the loopback: read-only, a frame that halts within a branch of a partition is not given; the store that split keeps the join, the rest of that branch alone executed by the loopback, writing, within the same transaction, and joins what every branch ends with, going on after the join; as a full run

In [ ]:
one, two = made(), made()


def split(Person, Book):
    """Each person x, then a partition: its first branch makes x 36, its second binds whom x knows; then the book."""
    arm = lambda then: lambda e: e.branch(lambda b: b.then([then]))
    return lambda txn: txn.uid("split/1").mutations([lambda mu: mu.map(lambda m: m.symbols([
        lambda p: p.symbols({"x": Person})]).then([lambda w: w.partition(lambda q: q.branches([
            arm(lambda w2: w2.update(lambda u: u.apply([lambda p: p.symbols({"x": Person}).spec([lambda _: _.x.age(36)])]))),
            arm(lambda w2: w2.map(lambda m2: m2.symbols([lambda p: p.symbols({"x": Person, "k": Person}).spec([
                lambda _: _.x.knows([_.k])])]))),
        ]).then([lambda w3: w3.map(lambda m3: m3.symbols([lambda p: p.symbols({"bk": Book})]))]))]))])


splitting = lambda t: split(t.crm.Person, t.crm.Book)(t)
with one.transaction() as txn:
    items = list(txn.exec(Frame(), splitting, False))
    assert read_people(one) == [("Ada", 36, ["Bob"]), ("Bob", 36, [])]  # the loopback's write, in this transaction
given = frames(items)  # Ada holds the first branch; Bob halted in it, his rest executed by the loopback, and joined
assert all(frame.remainder is None and frame.scope == "0" for frame in given) and items[-1] == {"end": "0"}
full = two.transaction(splitting)
shown = lambda store, given_: sorted(sorted((s, v) for s, v in each.items() if s != "bk") for each in names(store, [
    {s: t for s, t in frame.items() if s != "bk"} for frame in given_]))
assert shown(one, given) == shown(two, full) == [[("k", "Bob"), ("x", "Ada")], [("x", "Ada")], [("x", "Bob")]]
assert all("bk" in frame for frame in given)  # the terms after the join, gone on with
assert people(one) == people(two) == [("Ada", 36, ["Bob"]), ("Bob", 36, [])]  # committed once, with the scope

<a id="EXE-05"></a>
## EXE-05 · A branch's completion marker: a frame sent below with the rest of a branch carries the branch's id, its scope; executed alone, the rest of the branch and not the terms after the partition, its frames carry that id and end with its marker

In [ ]:
one = made()
expression = one.Mutations.OfTransaction(split(one.registered("crm.Person"), one.registered("crm.Book")))
rest = Remainder(expression, ("mutations", 0, "map", "then", 0, "partition", "branches", 0, "then", 0, "update"))
bob = one.transaction(ada_then(met("Bob")))[0]["y"]
items = run(one, Frame({"x": bob}, scope="0/0"), rest, True)
assert [item.scope if isinstance(item, Frame) else item for item in items] == ["0/0", {"end": "0/0"}]
assert list(items[0]) == ["x"] and people(one) == [("Ada", 36, ["Bob"]), ("Bob", 36, [])]  # no 'bk': the branch alone

<a id="EXE-06"></a>
## EXE-06 · Errors annotated in the stream: a frame a term fails to process is given annotated, the others too, nothing raised by `exec`, whether writing or read-only, or in the rest of a branch the loopback executes; the scope then commits nothing, raising the first error annotated, as `store.transaction` does; an input frame whose reference designates nothing, or that binds a counted object of another transaction, is given back annotated, and one given annotated as it is, executed no further

In [ ]:
one = made()
with one.transaction():  # a spec naming Bob by his path, built while he is held
    plain = Plain.ToPlain(one).Transaction(one.Mutations.OfTransaction(lambda t: t.uid("knows/1").mutations([
        lambda mu: mu.map(lambda m: m.symbols([lambda p: p.symbols({"x": one.crm.Person})]).then([
            lambda w: w.partition(lambda q: q.branches([lambda e: e.branch(lambda b: b.then([
                lambda w2: w2.update(lambda u: u.apply([lambda p: p.symbols({"x": one.crm.Person}).spec([
                    lambda _: _.x.age(1)])]).then([
                    lambda w3: w3.update(lambda u3: u3.apply([lambda p: p.symbols({"x": one.crm.Person}).spec([
                        lambda _: _.x.knows([one.singleton("crm.TheBook").contacts({"key": "bob"})])])]))]))]))]))]))])))
knowing = Plain.FromPlain(one).Transaction(plain)
bob = one.transaction(ada_then(met("Bob")))[0]["y"]
with one.transaction():
    one.crm.Book(one.singleton("crm.TheBook"), lambda b: b.contacts(lambda a: a.remove(lambda e: e.key("bob"))))
seen = []
with raises(LookupError, match="no object of the store has the path a spec refers to"):
    with one.transaction() as txn:
        seen += list(txn.exec(Frame(), knowing, True))  # nothing raised here: the frame is annotated
(failed,) = frames(seen)  # Ada, the one person left: Bob's path names nothing
assert names(one, [failed]) == [{"x": "Ada"}] and seen[1] == {"end": "0"}
assert [(e.subterm, e.kind, e.message) for e in failed.errors] == [(
    "$.mutations[0].map.then[0].partition.branches[0].then[0].update.then[0].update", "LookupError",
    "no object of the store has the path a spec refers to")]
assert people(one) == [("Ada", 36, [])]  # uncommitted, whole: Ada's age not written
with raises(LookupError, match="no object of the store has the path a spec refers to"):
    run(one, Frame(), knowing, False)  # read-only, in the rest of the branch the loopback executes
with raises(LookupError, match="no object of the store has the path a spec refers to"):
    one.transaction(knowing)
rest = Remainder(knowing, ("mutations", 0, "map", "then", 0, "partition", "branches", 0, "then", 0, "update", "then", 0,
                           "update"))
ada = one.transaction(ada_then(aged(36)))[0]
with raises(LookupError, match="no object of the store has the path a spec refers to"):  # the first annotated
    with one.transaction() as txn:
        (failed,) = frames(txn.exec(ada, rest, False))  # read-only, a term that fails before it would write
        (dangling,) = frames(txn.exec({"x": bob}, rest, True))
        (again,) = frames(txn.exec(failed, rest, True))  # annotated: given back as it is
assert failed.errors[0].kind == "LookupError" and failed.remainder is None
assert [(e.subterm, e.kind, e.message) for e in dangling.errors] == [(rest.subterm, "LookupError",
    "symbol 'x' of the frame: no object of the store has its reference's path")] and list(dangling) == []
assert again.errors == failed.errors
noted = one.transaction(lambda t: t.mutations([lambda mu: mu.upsert(lambda u: u.exists([
    lambda p: p.symbols({"n": t.crm.Note}).spec([lambda _: _.n.text("x")])]))]))[0]
with raises(LookupError):
    with one.transaction() as txn:
        (refused,) = frames(txn.exec(noted, rest, True))  # its counted object, another transaction's
assert refused.errors[0].message.startswith("symbol 'n' of the frame: a counted 'crm.Note' has no reference")

<a id="EXE-07"></a>
## EXE-07 · An input stream consumed as it arrives: a generator's frames, each executed as it is taken, its frames given before the next is taken; a stream left before its end leaves what it wrote to the transaction; an error of no one frame is raised, carrying the frames given

In [ ]:
one = made()
everyone = one.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([
    lambda p: p.symbols({"x": t.crm.Person})]))]))
older = lambda t: t.uid("older/2").mutations([lambda mu: mu.update(lambda u: u.apply([
    lambda p: p.symbols({"x": t.crm.Person}).spec([lambda _: _.x.age(60)])]))])
log = []


def arriving():
    for frame in everyone:
        log.append(("taken", frame.scope))
        yield Frame({"x": frame["x"]})


with one.transaction() as txn:
    for item in txn.exec(arriving(), older, True):
        log.append(("given", item.scope) if isinstance(item, Frame) else ("marker", item["end"]))
assert log == [("taken", "0"), ("given", "0"), ("marker", "0"), ("taken", "0"), ("given", "1"), ("marker", "1")]
assert people(one) == [("Ada", 60, ["Bob"]), ("Bob", 60, [])]
with one.transaction() as txn:
    stream = txn.exec(arriving(), lambda t: t.uid("older/3").mutations([lambda mu: mu.update(lambda u: u.apply([
        lambda p: p.symbols({"x": t.crm.Person}).spec([lambda _: _.x.age(70)])]))]), True)
    assert isinstance(next(stream), Frame)
    stream.close()  # left after Ada's: what it wrote is the transaction's, which commits
assert people(one) == [("Ada", 70, ["Bob"]), ("Bob", 60, [])]
try:  # the second input frame binds no x
    with one.transaction() as txn:
        list(txn.exec([Frame({"x": everyone[0]["x"]}), Frame()], older, True))
    raise AssertionError("expected an error")
except ValueError as error:
    assert str(error) == "symbol 'x' of an update is bound by no term before it"
    assert names(one, error.frames) == [{"x": "Ada"}]  # the frames given before it
assert people(one) == [("Ada", 70, ["Bob"]), ("Bob", 60, [])]  # the scope raised: nothing committed

<a id="EXE-08"></a>
## EXE-08 · A frame's journal: the changes made in interpreting its input frame, a `Changes.Journal` of one transaction, its id the transaction's uid and the frame's position in the stream, carried by the first frame the input frame gives, none by the others nor by a frame that changed nothing, a halted one among them; the loopback's frames carry theirs; applied by `Changes.apply`, the journals make the same changes in another store

In [ ]:
one, two = made(), made()
Journal = one.registered("Changes.Journal")
journals = lambda store, given: [None if frame.journal is None else Plain.ToPlain(store)(Journal, frame.journal)
                                  for frame in given]
everyone = one.transaction(lambda t: t.uid("everyone/1").mutations([lambda mu: mu.map(lambda m: m.symbols([
    lambda p: p.symbols({"x": t.crm.Person})]))]))
assert journals(one, everyone) == [None, None]  # it changed nothing
older = lambda t: t.uid("older/8").mutations([lambda mu: mu.update(lambda u: u.apply([
    lambda p: p.symbols({"x": t.crm.Person}).spec([lambda _: _.x.age(60)])]))])
with one.transaction() as txn:
    given = frames(txn.exec([{"x": frame["x"]} for frame in everyone], older, True))
written = journals(one, given)
assert [transaction["id"] for (transaction,) in written] == ["older/8#0", "older/8#1"]
assert written[0][0]["changes"] == [{"property": {"holder": "o0", "name": "age", "mutations": [{"set": {"value": 60}}]}}]
cy = lambda t: t.uid("cy/8").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p.symbols(
    {"c": t.crm.Person, "book": t.crm.Book}).spec([lambda _: _.book.contacts({"cy": lambda e: e.contact(_.c)}),
                                                  lambda _: _.c.name("Cy")])]).then([
    lambda w: w.map(lambda m: m.symbols([lambda p: p.symbols({"y": t.crm.Person})]))]))])
made_cy = one.transaction(cy)  # one input frame, three frames: the first carries what it changed
assert [frame.journal is None for frame in made_cy] == [False, True, True]
((created,),) = journals(one, made_cy[:1])
assert created["id"] == "cy/8#0" and [list(change) for change in created["changes"]] == [["entry"], ["property"]]
with one.transaction() as txn:
    (halted,) = frames(txn.exec(Frame(), ada_then(aged(61)), False))  # read-only: it halts, having changed nothing
    (looped,) = frames(txn.exec(halted, halted.remainder, True))  # the loopback's frame carries what it changed
assert halted.journal is None and journals(one, [looped])[0][0]["id"] == "ada/1#1"
two.transaction(lambda t: Changes.apply(t, Plain.FromPlain(two)(two.registered("Changes.Journal"), [
    transaction for journal in journals(one, [*given, *made_cy, looped]) if journal for transaction in journal])))
assert people(two) == people(one) == [("Ada", 61, ["Bob"]), ("Bob", 60, []), ("Cy", None, [])]